# SensiFake: final ordinal sensitivity experiment V3

This local notebook runs one experiment from the trained V2 ResNet50 backbone. The new two-output head learns `P(score >= 1)` and `P(score >= 2)` with weighted ordinal BCE. Human targets and existing assignments are read from the canonical 961-image manifest. The 140-image frozen test is evaluated once, after all training and validation threshold decisions.

All generated artifacts stay under `model_autoannotator/output/`. Historical annotations, manifests, checkpoints, outputs, and the V2 final dataset are read only. A fresh run refuses to overwrite an existing V3 experiment.

Run from the repository root using the existing GPU virtual environment:

```bash
source .venv/bin/activate
python -m pip install nbconvert ipykernel  # only these notebook packages are missing locally
python -m ipykernel install \
  --prefix "$PWD/model_autoannotator/output/metrics/jupyter" \
  --name sensifake-v3 --display-name "SensiFake V3"
SENSIFAKE_V3_MODE=run \
JUPYTER_PATH="$PWD/model_autoannotator/output/metrics/jupyter/share/jupyter${JUPYTER_PATH:+:$JUPYTER_PATH}" \
python -m nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=sensifake-v3 \
  "model_autoannotator/script/sensitivity_ordinal_v3.ipynb"
```

`SENSIFAKE_V3_MODE=preflight` executes integrity/model/forward checks only. Default mode `run` trains, calibrates, evaluates, compares, plots, and exports the corpus only if V3 wins. Set `SENSIFAKE_V3_EXPORT_IF_WINS=0` to defer that optional export. CUDA is detected automatically; CPU is used only when CUDA is unavailable. Dependencies: the existing compatible torch/torchvision, numpy, pandas, Pillow, scikit-learn and matplotlib, plus nbconvert/ipykernel for terminal execution. No package installation occurs inside this notebook.

**Fixed decisions before test evaluation:** checkpoint selection uses validation macro-F1 with thresholds `(0.5, 0.5)`; afterward the selected checkpoint alone is calibrated over all 441 threshold pairs in `[0.30, 0.70]` with step `0.02`. Calibration ties prefer thresholds closest to `(0.5, 0.5)`, then ascending t1/t2. Monotonic correction keeps `p_ge_1` and replaces `p_ge_2` with `min(p_ge_1, p_ge_2)`; raw probabilities and violation counts are saved.

**Prespecified final selection:** V3 must strictly exceed V2 frozen-test macro-F1. A decrease exceeding 0.05 in HIGH F1, HIGH recall, balanced accuracy, or any source macro-F1, or an ordinal-MAE increase exceeding 0.02, vetoes promotion. These conservative regression limits are fixed now and never tuned on test results.

In [1]:
import os
import sys
from pathlib import Path

RUN_MODE = os.environ.get("SENSIFAKE_V3_MODE", "run")
if RUN_MODE not in {"run", "preflight"}:
    raise ValueError("SENSIFAKE_V3_MODE must be run or preflight")
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "model_autoannotator/inputs/v2/human_review_retraining_v2_manifest.csv").is_file()
                  and (p / "model_autoannotator/script/sensitivity_v2_booster.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from within the SensiFake repository")
V3_ROOT = REPO_ROOT / "model_autoannotator"
OUTPUT = V3_ROOT / "output"
DIRECTORIES = {name: OUTPUT / name for name in
               ("checkpoints", "metrics", "plots", "predictions", "final_dataset")}
for directory in DIRECTORIES.values():
    directory.mkdir(parents=True, exist_ok=True)
# Matplotlib cache is kept within the V3 plot directory too.
os.environ["MPLCONFIGDIR"] = str(DIRECTORIES["plots"] / ".matplotlib_cache")
sys.path.insert(0, str(REPO_ROOT))

import csv
import json
import random
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torchvision
import sklearn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
from model_autoannotator.script.sensitivity_v2_booster import preflight
from model_autoannotator.script.build_human_review_retraining_v2_manifest import (
    MANIFEST, PACKAGE, REVIEW, MASTER, CLASSES, require, read_csv, sha256,
)

SEED = 42
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASSES)}
MODEL_VERSION = "resnet50_layer4_fc_weighted_ordinal_v3"
V1_CHECKPOINT = REPO_ROOT / "model_autoannotator/notebooks/experiments/final output/best_model.pt"
V2_CHECKPOINT = REPO_ROOT / "model_autoannotator/notebooks/experiments/sensitivity_resnet50_v2_output/best_model_boosted.pt"
BEST_CHECKPOINT = DIRECTORIES["checkpoints"] / "best_model_ordinal_v3.pt"
DATA_ROOT = REPO_ROOT / PACKAGE
BATCH_SIZE = 32
NUM_WORKERS = 2
MAX_EPOCHS = 12
PATIENCE = 5
LAYER4_LR = 2e-6
HEAD_LR = 2e-5
WEIGHT_DECAY = 1e-4
DEFAULT_THRESHOLDS = (0.5, 0.5)
THRESHOLD_GRID = np.round(np.arange(0.30, 0.701, 0.02), 2)
EXPORT_IF_V3_WINS = os.environ.get("SENSIFAKE_V3_EXPORT_IF_WINS", "1") != "0"
SELECTION_POLICY = {"min_macro_f1_gain": 0.0, "max_high_f1_drop": 0.05,
                    "max_high_recall_drop": 0.05, "max_balanced_accuracy_drop": 0.05,
                    "max_source_macro_f1_drop": 0.05, "max_ordinal_mae_increase": 0.02}
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}; GPU: {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'unavailable, CPU fallback'}", flush=True)
print(f"PyTorch {torch.__version__}; torchvision {torchvision.__version__}", flush=True)
print(f"Mode: {RUN_MODE}; outputs: {OUTPUT}", flush=True)


def save_json(path, value):
    with Path(path).open("x", encoding="utf-8") as stream:
        stream.write(json.dumps(value, indent=2, allow_nan=False) + "\n")


def check_inputs_unchanged():
    for relative, digest in protected_sha256.items():
        require(sha256(REPO_ROOT / relative) == digest, f"Protected input changed: {relative}")

Device: cuda; GPU: NVIDIA GeForce RTX 4060 Laptop GPU


PyTorch 2.11.0+cu128; torchvision 0.26.0+cu128


Mode: run; outputs: /home/sarah/Desktop/University/ComputerVision/SensiFake/autoannotator-final version/output


## Preflight and locked human-supervised data

Reuse the existing V2 integrity checks, including original assignments, authoritative `review_sara.csv` targets, source authenticity metadata, SHA-256 identities, and image decoding. No splitting or pseudo-label generation occurs. Test labels are read only for integrity until the final evaluation stage.

In [2]:
rows, test_rows, preflight_report = preflight(REPO_ROOT, verify_images=True)
train_frame = pd.DataFrame(rows)
test_frame = pd.DataFrame(test_rows)
fit_frame = train_frame.loc[train_frame.internal_split == "effective_train"].copy()
validation_frame = train_frame.loc[train_frame.internal_split == "validation"].copy()
require((len(train_frame), len(fit_frame), len(validation_frame), len(test_frame)) == (961, 769, 192, 140),
        "Unexpected fixed split sizes")
require(set(fit_frame.content_hash).isdisjoint(validation_frame.content_hash), "Train/validation overlap")
require(set(train_frame.content_hash).isdisjoint(test_frame.content_hash), "Frozen-test overlap")
require(set(train_frame.image_id).isdisjoint(test_frame.image_id), "Frozen-test stable-ID overlap")
require(all(r["source_label"] in {"real", "fake"} for r in rows + test_rows), "Invalid authenticity metadata")
# Explicitly verify both original and reviewed authenticity against original human sources.
master_by_hash = {r["content_hash"]: r for r in read_csv(REPO_ROOT / MASTER)}
reviews_by_hash = {r["content_hash"]: r for r in read_csv(REPO_ROOT / REVIEW)}
for row in rows + test_rows:
    reference = reviews_by_hash.get(row["content_hash"]) or master_by_hash[row["content_hash"]]
    require(row["source_label"] == reference["source_label"]
            and row["source_dataset"] == reference["source_dataset"], "Human source metadata mismatch")
    require(CLASS_TO_INDEX[row["final_sensitivity_level"]] in {0, 1, 2}, "Invalid ordinal target")
weights = Counter(float(r["sample_weight"]) for r in fit_frame.to_dict("records"))
require(weights == {1.0: 678, 2.0: 91}, "V2 corrective weight counts changed")
require(all(r["sample_weight"] == (2.0 if r["annotation_source"] == "human_review"
                                  and r["model_prediction"] != r["human_final_label"] else 1.0)
            for r in rows), "V2 corrective weighting mismatch")

protected_sha256 = dict(preflight_report["input_sha256"])
protected_paths = [V2_CHECKPOINT, V1_CHECKPOINT,
                   REPO_ROOT / "model_autoannotator/notebooks/experiments/sensitivity_resnet50_human_review_v2.ipynb",
                   REPO_ROOT / "model_autoannotator/notebooks/experiments/sensitivity_resnet50_human_review_v2_1.ipynb",
                   DATA_ROOT / "corpus_manifest.csv"]
for path in protected_paths:
    require(path.is_file(), f"Missing protected input: {path}")
    protected_sha256[path.relative_to(REPO_ROOT).as_posix()] = sha256(path)
if RUN_MODE == "run":
    for path in [BEST_CHECKPOINT, DIRECTORIES["metrics"] / "configuration.json",
                 DIRECTORIES["metrics"] / "test_metrics.json"]:
        require(not path.exists(), f"V3 experiment already exists; refusing to overwrite: {path}")
print("Data checks passed:", {"total": 961, "training": 769, "validation": 192,
      "frozen_test": 140, "reviewed_training": 320, "reviewed_validation": 80,
      "training_weight_counts": dict(weights), "train_test_overlap": 0, "validation_test_overlap": 0}, flush=True)

Data checks passed: {'total': 961, 'training': 769, 'validation': 192, 'frozen_test': 140, 'reviewed_training': 320, 'reviewed_validation': 80, 'training_weight_counts': {1.0: 678, 2.0: 91}, 'train_test_overlap': 0, 'validation_test_overlap': 0}


## V2 transforms and dataloaders

Augmentation, resolution, normalization, EXIF handling and batch size are identical to V2. Validation and test have no training augmentation.

In [3]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.90, 1.00), ratio=(0.95, 1.05),
        interpolation=transforms.InterpolationMode.BILINEAR, antialias=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.10, contrast=0.10,
                                                  saturation=0.05, hue=0.0)], p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
eval_transform = ResNet50_Weights.IMAGENET1K_V2.transforms()

class LabeledImages(Dataset):
    def __init__(self, frame, transform):
        self.rows = frame.to_dict("records")
        self.transform = transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(REPO_ROOT / row["image_path"]) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
        return (self.transform(image), CLASS_TO_INDEX[row["final_sensitivity_level"]],
                float(row.get("sample_weight", 1.0)), index)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(frame, *, training=False):
    return DataLoader(LabeledImages(frame, train_transform if training else eval_transform),
        batch_size=BATCH_SIZE, shuffle=training, num_workers=NUM_WORKERS,
        worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(SEED),
        pin_memory=DEVICE.type == "cuda")

fit_loader = make_loader(fit_frame, training=True)
validation_loader = make_loader(validation_frame)
# No test dataloader iteration occurs until after checkpoint and thresholds are locked.

## Trained V2 backbone, fresh ordinal head, frozen BatchNorm

Strictly load the complete V2 3-class checkpoint first to verify architecture and mapping. Then discard its fc head and initialize a new `Linear(2048, 2)` head with seed 42. Only layer4 convolution parameters and the new head are optimized. BatchNorm affine parameters and running buffers are checked throughout training.

In [4]:
v2_checkpoint = torch.load(V2_CHECKPOINT, map_location="cpu", weights_only=True)
require(tuple(v2_checkpoint.get("class_order", ())) == CLASSES, "V2 class-index mapping mismatch")
require(v2_checkpoint.get("model_version") == "resnet50_layer4_fc_human_review_corrective_booster_v2", "Wrong V2 checkpoint version")
require(v2_checkpoint.get("manifest_sha256") == sha256(REPO_ROOT / MANIFEST), "V2 manifest ancestry mismatch")
require(v2_checkpoint.get("initial_checkpoint_sha256") == sha256(V1_CHECKPOINT), "V2 V1 ancestry mismatch")
model = resnet50(weights=None)
require(model.fc.in_features == 2048, "Unexpected ResNet50 architecture")
model.fc = nn.Linear(2048, 3)
model.load_state_dict(v2_checkpoint["model_state_dict"], strict=True)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
model.fc = nn.Linear(2048, 2)  # V2 classification head is deliberately discarded.
for parameter in model.parameters():
    parameter.requires_grad_(False)
for parameter in model.layer4.parameters():
    parameter.requires_grad_(True)
for parameter in model.fc.parameters():
    parameter.requires_grad_(True)


def freeze_batchnorm():
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            for parameter in module.parameters():
                parameter.requires_grad_(False)

freeze_batchnorm()
expected_names = {f"layer4.{block}.conv{conv}.weight" for block in range(3) for conv in range(1, 4)} | {
    "layer4.0.downsample.0.weight", "fc.weight", "fc.bias"}
trainable_names = {name for name, parameter in model.named_parameters() if parameter.requires_grad}
require(trainable_names == expected_names, "Unexpected V3 trainable parameters")
for name, value in model.state_dict().items():
    if not name.startswith("fc."):
        require(torch.equal(value, v2_checkpoint["model_state_dict"][name]), f"V2 backbone mismatch: {name}")
model = model.to(DEVICE)
bn_buffers = {name: value.detach().cpu().clone() for name, value in model.named_buffers()}


def assert_batchnorm_frozen():
    for name, module in model.named_modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            require(not module.training and all(not p.requires_grad for p in module.parameters()),
                    f"BatchNorm train state/affine drift: {name}")
    for name, value in model.named_buffers():
        require(torch.equal(value.detach().cpu(), bn_buffers[name]), f"BatchNorm running-buffer drift: {name}")

model.eval()
with torch.inference_mode():
    smoke_logits = model(torch.zeros(1, 3, 224, 224, device=DEVICE))
require(tuple(smoke_logits.shape) == (1, 2) and bool(torch.isfinite(smoke_logits).all()), "Ordinal forward failed")
assert_batchnorm_frozen()
parameter_report = {"trainable_names": sorted(trainable_names),
    "layer4_parameters": sum(p.numel() for p in model.layer4.parameters() if p.requires_grad),
    "ordinal_head_parameters": sum(p.numel() for p in model.fc.parameters()),
    "frozen": ["conv1", "bn1", "layer1", "layer2", "layer3", "all BatchNorm affine parameters and running statistics"]}
preflight_report.update({"model_version": MODEL_VERSION, "device": str(DEVICE),
    "checkpoint_load": "V2 strict load; exact backbone transfer; fresh 2-output head",
    "checkpoint_sha256": sha256(V2_CHECKPOINT), "parameters": parameter_report,
    "forward_smoke_test": "passed", "training_started": False})
print("Model preflight passed:", json.dumps(parameter_report, indent=2), flush=True)
if RUN_MODE == "preflight":
    print("Preflight complete. No training, calibration or test inference performed.", flush=True)

Model preflight passed: {
  "trainable_names": [
    "fc.bias",
    "fc.weight",
    "layer4.0.conv1.weight",
    "layer4.0.conv2.weight",
    "layer4.0.conv3.weight",
    "layer4.0.downsample.0.weight",
    "layer4.1.conv1.weight",
    "layer4.1.conv2.weight",
    "layer4.1.conv3.weight",
    "layer4.2.conv1.weight",
    "layer4.2.conv2.weight",
    "layer4.2.conv3.weight"
  ],
  "layer4_parameters": 14942208,
  "ordinal_head_parameters": 4098,
  "frozen": [
    "conv1",
    "bn1",
    "layer1",
    "layer2",
    "layer3",
    "all BatchNorm affine parameters and running statistics"
  ]
}


## Ordinal loss, monotonic decoding and shared evaluation conventions

Targets are LOW `[0,0]`, MEDIUM `[1,0]`, HIGH `[1,1]`. The two unreduced BCE losses are averaged per image, multiplied by the V2 corrective weight (1 or 2), and averaged over the batch. There are no class weights or sampling changes.

In [5]:
def ordinal_targets(scores):
    return torch.stack((scores >= 1, scores >= 2), dim=1).to(dtype=torch.float32)


def ordinal_loss(logits, scores, sample_weights):
    per_threshold = nn.functional.binary_cross_entropy_with_logits(logits, ordinal_targets(scores), reduction="none")
    per_sample = per_threshold.mean(dim=1)
    return (per_sample * sample_weights.to(device=logits.device, dtype=logits.dtype)).mean()


def monotonic_probabilities(raw_probabilities):
    corrected = np.asarray(raw_probabilities, dtype=float).copy()
    require(corrected.ndim == 2 and corrected.shape[1] == 2 and np.isfinite(corrected).all(), "Invalid ordinal probabilities")
    require(((corrected >= 0) & (corrected <= 1)).all(), "Ordinal probability outside [0,1]")
    violations = corrected[:, 1] > corrected[:, 0]
    corrected[:, 1] = np.minimum(corrected[:, 1], corrected[:, 0])
    return corrected, violations


def decode(probabilities, t1=0.5, t2=0.5):
    require(0 < t1 < 1 and 0 < t2 < 1, "Invalid thresholds")
    corrected, _ = monotonic_probabilities(probabilities)
    return np.where(corrected[:, 0] < t1, 0, np.where(corrected[:, 1] < t2, 1, 2)).astype(int)


def apply_thresholds(frame, thresholds):
    frame = frame.copy()
    scores = decode(frame[["p_ge_1", "p_ge_2"]].to_numpy(), *thresholds)
    frame["predicted_sensitivity_score"] = scores
    frame["predicted_sensitivity_label"] = [CLASSES[s] for s in scores]
    return frame


def predict_labeled(frame, loader):
    model.eval()
    records = []
    source_rows = frame.to_dict("records")
    with torch.inference_mode():
        for images, scores, _, indices in loader:
            raw = torch.sigmoid(model(images.to(DEVICE, non_blocking=True))).cpu().numpy()
            corrected, violations = monotonic_probabilities(raw)
            for j, index in enumerate(indices.tolist()):
                row = source_rows[index]
                records.append({"image_id": row["image_id"], "content_hash": row["content_hash"],
                    "image_path": row["image_path"], "source_dataset": row["source_dataset"],
                    "real_fake_label": 0 if row["source_label"] == "real" else 1,
                    "true_sensitivity_score": int(scores[j]), "true_sensitivity_label": CLASSES[int(scores[j])],
                    "p_ge_1_raw": float(raw[j, 0]), "p_ge_2_raw": float(raw[j, 1]),
                    "p_ge_1": float(corrected[j, 0]), "p_ge_2": float(corrected[j, 1]),
                    "monotonicity_corrected": bool(violations[j])})
    require(len(records) == len(frame), "Incomplete ordinal predictions")
    return pd.DataFrame(records)


def group_metrics(frame):
    true = frame.true_sensitivity_score.to_numpy(dtype=int)
    predicted = frame.predicted_sensitivity_score.to_numpy(dtype=int)
    require(len(true) > 0, "Cannot evaluate empty group")
    precision, recall, f1, support = precision_recall_fscore_support(true, predicted, labels=[0, 1, 2], zero_division=0)
    return {"support": len(true), "class_order": list(CLASSES),
        "accuracy": float(accuracy_score(true, predicted)),
        "macro_f1": float(f1_score(true, predicted, labels=[0, 1, 2], average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(true, predicted)),
        "ordinal_mae": float(np.mean(np.abs(true - predicted))),
        "high_recall": float(recall[2]), "high_f1": float(f1[2]),
        "per_class": {c: {"precision": float(precision[i]), "recall": float(recall[i]),
                           "f1": float(f1[i]), "support": int(support[i])} for i, c in enumerate(CLASSES)},
        "confusion_matrix": confusion_matrix(true, predicted, labels=[0, 1, 2]).tolist(),
        "monotonicity_corrections": int(frame.monotonicity_corrected.sum()) if "monotonicity_corrected" in frame else None}


def classification_metrics(frame):
    result = group_metrics(frame)
    result["by_source_dataset"] = {source: group_metrics(group) for source, group in frame.groupby("source_dataset", sort=True)}
    return result


def calibrate_thresholds(frame):
    grid_records = []
    for t1 in THRESHOLD_GRID:
        for t2 in THRESHOLD_GRID:
            values = group_metrics(apply_thresholds(frame, (float(t1), float(t2))))
            grid_records.append({"t1": float(t1), "t2": float(t2), **{k: values[k] for k in
                ("accuracy", "macro_f1", "balanced_accuracy", "ordinal_mae", "high_recall", "high_f1")}})
    grid = pd.DataFrame(grid_records)
    grid["distance_from_default"] = abs(grid.t1 - 0.5) + abs(grid.t2 - 0.5)
    best = grid.sort_values(["macro_f1", "distance_from_default", "t1", "t2"],
                           ascending=[False, True, True, True], kind="stable").iloc[0]
    return (float(best.t1), float(best.t2)), grid

## Verify existing V1/V2 frozen-test results

Reuse saved baseline metrics and predictions; verify their checkpoint checksums, frozen-test checksum, identities, targets, source memberships, and recomputed metrics from the saved predictions. This never runs baseline inference and never supplies baseline test results to training or calibration.

In [6]:
def load_baseline(name, directory, checkpoint_path):
    metrics_path = directory / "test_metrics.json"
    predictions_path = directory / "test_predictions.csv"
    require(metrics_path.is_file() and predictions_path.is_file(), f"Missing existing {name} evaluation")
    saved = json.loads(metrics_path.read_text())
    require(saved["support"] == 140 and saved["class_order"] == list(CLASSES), f"{name} class/support mismatch")
    require(saved["fixed_test_csv_sha256"] == sha256(DATA_ROOT / "test.csv"), f"{name} test checksum mismatch")
    require(saved["checkpoint_sha256"] == sha256(checkpoint_path), f"{name} checkpoint checksum mismatch")
    predictions = pd.read_csv(predictions_path)
    expected = {r["content_hash"]: (r["final_sensitivity_level"], r["source_dataset"]) for r in test_rows}
    require(len(predictions) == predictions.content_hash.nunique() == 140, f"{name} duplicate/incomplete predictions")
    require({r["content_hash"]: (r["true_sensitivity"], r["source_dataset"])
             for r in predictions.to_dict("records")} == expected, f"{name} different test images/targets")
    converted = predictions.rename(columns={"true_sensitivity": "true_sensitivity_label",
                                            "predicted_sensitivity": "predicted_sensitivity_label"}).copy()
    converted["true_sensitivity_score"] = converted.true_sensitivity_label.map(CLASS_TO_INDEX)
    converted["predicted_sensitivity_score"] = converted.predicted_sensitivity_label.map(CLASS_TO_INDEX)
    recalculated = classification_metrics(converted)
    for key in ("accuracy", "macro_f1", "balanced_accuracy", "ordinal_mae"):
        require(abs(saved[key] - recalculated[key]) < 1e-12, f"Stale {name} metric: {key}")
    for path in (metrics_path, predictions_path):
        protected_sha256[path.relative_to(REPO_ROOT).as_posix()] = sha256(path)
    return saved

BASELINE_DIRECTORY = REPO_ROOT / "model_autoannotator/notebooks/experiments/sensitivity_resnet50_v2_output"
baseline_metrics = {
    "V1": load_baseline("V1", BASELINE_DIRECTORY / "evaluate_v1", V1_CHECKPOINT),
    "V2": load_baseline("V2", BASELINE_DIRECTORY / "evaluate_v2", V2_CHECKPOINT),
}
print("Existing V1/V2 evaluation integrity checks passed; no baseline inference rerun.", flush=True)

Existing V1/V2 evaluation integrity checks passed; no baseline inference rerun.


## Short weighted ordinal fine-tuning

Twelve epochs maximum, patience five, AdamW with the V2 learning rates and weight decay. Save epoch zero first; replace the best checkpoint only for strict validation macro-F1 improvement. Validation has no gradient updates. Validation metrics below use the default `(0.5, 0.5)` thresholds throughout epoch selection.

In [7]:
if RUN_MODE == "run":
    configuration = {
        "model_version": MODEL_VERSION, "seed": SEED, "device": str(DEVICE),
        "gpu_name": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
        "runtime_versions": {"torch": str(torch.__version__), "torchvision": str(torchvision.__version__),
                             "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
        "manifest": MANIFEST.as_posix(), "manifest_sha256": sha256(REPO_ROOT / MANIFEST),
        "starting_checkpoint": V2_CHECKPOINT.relative_to(REPO_ROOT).as_posix(),
        "starting_checkpoint_sha256": sha256(V2_CHECKPOINT),
        "notebook_sha256": sha256(V3_ROOT / "script/sensitivity_ordinal_v3.ipynb"),
        "protected_input_sha256": protected_sha256,
        "training": 769, "validation": 192, "frozen_test": 140,
        "class_order": list(CLASSES), "ordinal_targets": {"low": [0, 0], "medium": [1, 0], "high": [1, 1]},
        "loss": "mean(mean(BCEWithLogitsLoss per threshold) per sample * V2 corrective sample weight)",
        "corrective_weights": {"original_v1": 1.0, "reviewed_v1_correct": 1.0, "reviewed_v1_error": 2.0},
        "training_weight_counts": {str(k): v for k, v in weights.items()}, "class_weights": None,
        "optimizer": "AdamW", "layer4_lr": LAYER4_LR, "ordinal_head_lr": HEAD_LR,
        "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
        "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "checkpoint_selection": "validation_macro_f1",
        "checkpoint_selection_thresholds": list(DEFAULT_THRESHOLDS),
        "threshold_grid": THRESHOLD_GRID.tolist(), "threshold_pairs": 441,
        "calibration_tie_break": "macro_f1 descending; L1 distance to defaults ascending; t1 then t2 ascending",
        "monotonic_rule": "keep raw p_ge_1; replace p_ge_2 with min(raw p_ge_1, raw p_ge_2)",
        "parameters": parameter_report, "transforms": "exact V2 training/evaluation transforms",
        "selection_policy": SELECTION_POLICY, "export_if_v3_wins": EXPORT_IF_V3_WINS,
    }
    save_json(DIRECTORIES["metrics"] / "configuration.json", configuration)
    save_json(DIRECTORIES["metrics"] / "preflight.json", preflight_report)
    fit_frame[["image_id", "content_hash", "source_dataset", "annotation_source", "human_final_label",
               "model_prediction", "final_sensitivity_level", "sample_weight"]].to_csv(
        DIRECTORIES["metrics"] / "training_sample_weights.csv", index=False)
    optimizer = torch.optim.AdamW([
        {"params": [p for p in model.layer4.parameters() if p.requires_grad], "lr": LAYER4_LR},
        {"params": list(model.fc.parameters()), "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    def save_candidate(epoch, values):
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch,
            "validation_macro_f1_default_thresholds": values["macro_f1"],
            "class_order": CLASSES, "ordinal_outputs": ("score_ge_1", "score_ge_2"),
            "model_version": MODEL_VERSION, "manifest_sha256": configuration["manifest_sha256"],
            "starting_checkpoint_sha256": configuration["starting_checkpoint_sha256"],
            "selection_thresholds": DEFAULT_THRESHOLDS}, BEST_CHECKPOINT)

    baseline_validation_predictions = apply_thresholds(predict_labeled(validation_frame, validation_loader), DEFAULT_THRESHOLDS)
    initial_validation_metrics = classification_metrics(baseline_validation_predictions)
    baseline_validation_predictions.to_csv(DIRECTORIES["predictions"] / "validation_epoch_zero.csv", index=False)
    save_json(DIRECTORIES["metrics"] / "validation_epoch_zero.json", initial_validation_metrics)
    best_score = initial_validation_metrics["macro_f1"]
    best_epoch, stale_epochs = 0, 0
    save_candidate(0, initial_validation_metrics)
    history = [{"epoch": 0, "train_loss": None, **{
        f"validation_{key}": initial_validation_metrics[key] for key in
        ("accuracy", "macro_f1", "balanced_accuracy", "ordinal_mae", "high_recall", "high_f1")}}]
    print(f"Epoch 0 validation macro-F1={best_score:.6f}", flush=True)
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        freeze_batchnorm()
        assert_batchnorm_frozen()
        total_loss, sample_count = 0.0, 0
        for images, scores, sample_weights, _ in fit_loader:
            images = images.to(DEVICE, non_blocking=True)
            scores = scores.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = ordinal_loss(model(images), scores, sample_weights)
            require(bool(torch.isfinite(loss)), "Non-finite ordinal training loss")
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach().cpu()) * len(scores)
            sample_count += len(scores)
        assert_batchnorm_frozen()
        validation_predictions = apply_thresholds(predict_labeled(validation_frame, validation_loader), DEFAULT_THRESHOLDS)
        values = classification_metrics(validation_predictions)
        history.append({"epoch": epoch, "train_loss": total_loss / sample_count,
                        **{f"validation_{key}": values[key] for key in
                           ("accuracy", "macro_f1", "balanced_accuracy", "ordinal_mae", "high_recall", "high_f1")}})
        with (DIRECTORIES["metrics"] / "validation_history.jsonl").open("a", encoding="utf-8") as stream:
            stream.write(json.dumps({"epoch": epoch, **values}) + "\n")
        if values["macro_f1"] > best_score + 1e-12:
            best_score, best_epoch, stale_epochs = values["macro_f1"], epoch, 0
            save_candidate(epoch, values)
        else:
            stale_epochs += 1
        pd.DataFrame(history).to_csv(DIRECTORIES["metrics"] / "training_history.csv", index=False)
        print(f"Epoch {epoch}: loss={history[-1]['train_loss']:.5f}; val macro-F1={values['macro_f1']:.5f}; "
              f"HIGH recall={values['high_recall']:.5f}; best epoch={best_epoch}; patience={stale_epochs}/{PATIENCE}", flush=True)
        if stale_epochs >= PATIENCE:
            print(f"Early stopping after epoch {epoch}", flush=True)
            break
    check_inputs_unchanged()
    selected_checkpoint = torch.load(BEST_CHECKPOINT, map_location="cpu", weights_only=True)
    require(selected_checkpoint["epoch"] == best_epoch, "Selected epoch mismatch")
    model.load_state_dict(selected_checkpoint["model_state_dict"], strict=True)
    model.eval()
    assert_batchnorm_frozen()
    print(f"Selected V3 epoch {best_epoch}; validation macro-F1 at default thresholds={best_score:.6f}", flush=True)

Epoch 0 validation macro-F1=0.301851


Epoch 1: loss=0.66519; val macro-F1=0.28580; HIGH recall=0.00000; best epoch=0; patience=1/5


Epoch 2: loss=0.56483; val macro-F1=0.30705; HIGH recall=0.00000; best epoch=2; patience=0/5


Epoch 3: loss=0.52573; val macro-F1=0.35641; HIGH recall=0.00000; best epoch=3; patience=0/5


Epoch 4: loss=0.49492; val macro-F1=0.38026; HIGH recall=0.03571; best epoch=4; patience=0/5


...

Epoch 9: loss=0.37660; val macro-F1=0.56548; HIGH recall=0.25000; best epoch=9; patience=0/5


Epoch 10: loss=0.35955; val macro-F1=0.60924; HIGH recall=0.39286; best epoch=10; patience=0/5


Epoch 11: loss=0.34664; val macro-F1=0.60203; HIGH recall=0.35714; best epoch=10; patience=1/5


Epoch 12: loss=0.33224; val macro-F1=0.66269; HIGH recall=0.50000; best epoch=12; patience=0/5


Selected V3 epoch 12; validation macro-F1 at default thresholds=0.662686


## Validation-only threshold calibration

Search all threshold pairs on the selected model's 192 validation images. There are no test inputs to the calibration function. Only the thresholds change here; model weights and the chosen epoch stay fixed. Both cumulative probabilities are monotonic before hierarchical decoding, so every grid pair is a valid deterministic decoder.

In [8]:
if RUN_MODE == "run":
    selected_validation_raw = predict_labeled(validation_frame, validation_loader)
    calibrated_thresholds, calibration_grid = calibrate_thresholds(selected_validation_raw)
    require(len(calibration_grid) == 441, "Incomplete threshold grid")
    calibrated_validation = apply_thresholds(selected_validation_raw, calibrated_thresholds)
    calibrated_validation_metrics = classification_metrics(calibrated_validation)
    checkpoint_hash = sha256(BEST_CHECKPOINT)
    calibration_result = {"t1": calibrated_thresholds[0], "t2": calibrated_thresholds[1],
        "best_epoch": best_epoch, "checkpoint_sha256": checkpoint_hash,
        "validation_only": True, "validation_support": 192,
        "default_threshold_validation_macro_f1": best_score,
        "calibrated_validation_metrics": calibrated_validation_metrics,
        "monotonic_rule": configuration["monotonic_rule"], "tie_break": configuration["calibration_tie_break"]}
    save_json(DIRECTORIES["metrics"] / "threshold_calibration.json", calibration_result)
    calibration_grid.to_csv(DIRECTORIES["metrics"] / "threshold_calibration.csv", index=False)
    calibrated_validation.to_csv(DIRECTORIES["predictions"] / "validation_predictions.csv", index=False)
    save_json(DIRECTORIES["metrics"] / "validation_metrics.json", calibrated_validation_metrics)
    print("Validation calibration:", json.dumps(calibration_result, indent=2), flush=True)

Validation calibration: {
  "t1": 0.5,
  "t2": 0.44,
  "best_epoch": 12,
  "checkpoint_sha256": "48f6495d4a62ded10d98c3835ffcd33c8f5c653e5df19cb03aef62582060e9b4",
  "validation_only": true,
  "validation_support": 192,
  "default_threshold_validation_macro_f1": 0.6626861767388457,
  "calibrated_validation_metrics": {
    "support": 192,
    "class_order": [
      "low",
      "medium",
      "high"
    ],
    "accuracy": 0.7135416666666666,
    "macro_f1": 0.6854613668257453,
    "balanced_accuracy": 0.6743811382718344,
    "ordinal_mae": 0.2916666666666667,
    "high_recall": 0.6428571428571429,
    "high_f1": 0.6545454545454545,
    "per_class": {
      "low": {
        "precision": 0.7333333333333333,
        "recall": 0.5689655172413793,
        "f1": 0.6407766990291263,
        "support": 58
      },
      "medium": {
        "precision": 0.7166666666666667,
        "recall": 0.8113207547169812,
        "f1": 0.7610619469026548,
        "support": 106
      },
      "high": {
   

## One frozen-test evaluation and final selection

The test is first inferred here after all weights, epochs and thresholds are locked. V1/V2 metrics are reused from their verified saved results. V3 promotion follows the fixed rule stated above; there is no tuning or retry after viewing test results.

In [9]:
if RUN_MODE == "run":
    require(not (DIRECTORIES["metrics"] / "test_metrics.json").exists(), "V3 test already evaluated")
    require(sha256(BEST_CHECKPOINT) == checkpoint_hash, "Checkpoint changed after calibration")
    test_predictions = apply_thresholds(predict_labeled(test_frame, make_loader(test_frame)), calibrated_thresholds)
    require(len(test_predictions) == test_predictions.content_hash.nunique() == 140
            and set(test_predictions.content_hash) == set(test_frame.content_hash), "Frozen-test membership mismatch")
    test_metrics = classification_metrics(test_predictions)
    test_metrics.update({"checkpoint_sha256": checkpoint_hash, "model_version": MODEL_VERSION,
        "fixed_test_csv_sha256": sha256(DATA_ROOT / "test.csv"), "manifest_sha256": sha256(REPO_ROOT / MANIFEST),
        "epoch": best_epoch, "thresholds": {"t1": calibrated_thresholds[0], "t2": calibrated_thresholds[1]},
        "test_inference_passes": 1})
    save_json(DIRECTORIES["metrics"] / "test_metrics.json", test_metrics)
    test_predictions.to_csv(DIRECTORIES["predictions"] / "test_predictions.csv", index=False)
    comparison_metrics = {**baseline_metrics, "V3": test_metrics}
    vetoes = []
    v2 = baseline_metrics["V2"]
    macro_gain = test_metrics["macro_f1"] - v2["macro_f1"]
    if macro_gain <= SELECTION_POLICY["min_macro_f1_gain"] + 1e-12:
        vetoes.append("V3 does not strictly improve frozen-test macro-F1")
    for metric, policy_key in (("f1", "max_high_f1_drop"), ("recall", "max_high_recall_drop")):
        if v2["per_class"]["high"][metric] - test_metrics["per_class"]["high"][metric] > SELECTION_POLICY[policy_key] + 1e-12:
            vetoes.append(f"HIGH {metric} regression exceeds prespecified limit")
    if v2["balanced_accuracy"] - test_metrics["balanced_accuracy"] > SELECTION_POLICY["max_balanced_accuracy_drop"] + 1e-12:
        vetoes.append("Balanced-accuracy regression exceeds prespecified limit")
    if test_metrics["ordinal_mae"] - v2["ordinal_mae"] > SELECTION_POLICY["max_ordinal_mae_increase"] + 1e-12:
        vetoes.append("Ordinal-MAE regression exceeds prespecified limit")
    for source, values in v2["by_source_dataset"].items():
        if values["macro_f1"] - test_metrics["by_source_dataset"][source]["macro_f1"] > SELECTION_POLICY["max_source_macro_f1_drop"] + 1e-12:
            vetoes.append(f"Source macro-F1 regression exceeds prespecified limit: {source}")
    final_model = "V2" if vetoes else "V3"
    final_checkpoint = V2_CHECKPOINT if vetoes else BEST_CHECKPOINT
    selection = {"selected_model": final_model, "selected_checkpoint": final_checkpoint.relative_to(REPO_ROOT).as_posix(),
        "checkpoint_sha256": sha256(final_checkpoint), "v3_minus_v2_macro_f1": macro_gain,
        "ordinal_thresholds": {"t1": calibrated_thresholds[0], "t2": calibrated_thresholds[1]} if final_model == "V3" else None,
        "calibration_file": (DIRECTORIES["metrics"] / "threshold_calibration.json").relative_to(REPO_ROOT).as_posix() if final_model == "V3" else None,
        "vetoes": vetoes, "policy": SELECTION_POLICY,
        "conclusion": ("Ordinal formulation did not qualify to replace V2" if vetoes else "V3 ordinal formulation outperformed V2 under the fixed selection rule")}
    save_json(DIRECTORIES["metrics"] / "final_model_selection.json", selection)
    comparison = {"models": comparison_metrics, "selection": selection, "fixed_test_images": 140,
        "test_caution": "This repeatedly studied 140-image test is used for model comparison/selection; HIGH support is only 9. No significance claim or test-driven retuning.",
        "v3_minus_v2": {key: test_metrics[key] - v2[key] for key in
                        ("accuracy", "macro_f1", "balanced_accuracy", "ordinal_mae")}}
    save_json(DIRECTORIES["metrics"] / "comparison_v1_v2_v3.json", comparison)
    print("Final selection:", json.dumps(selection, indent=2), flush=True)

Final selection: {
  "selected_model": "V3",
  "selected_checkpoint": "autoannotator-final version/output/checkpoints/best_model_ordinal_v3.pt",
  "checkpoint_sha256": "48f6495d4a62ded10d98c3835ffcd33c8f5c653e5df19cb03aef62582060e9b4",
  "v3_minus_v2_macro_f1": 0.03443463674861147,
  "ordinal_thresholds": {
    "t1": 0.5,
    "t2": 0.44
  },
  "calibration_file": "autoannotator-final version/output/metrics/threshold_calibration.json",
  "vetoes": [],
  "policy": {
    "min_macro_f1_gain": 0.0,
    "max_high_f1_drop": 0.05,
    "max_high_recall_drop": 0.05,
    "max_balanced_accuracy_drop": 0.05,
    "max_source_macro_f1_drop": 0.05,
    "max_ordinal_mae_increase": 0.02
  },
  "conclusion": "V3 ordinal formulation outperformed V2 under the fixed selection rule"
}


## Report plots and numerical tables

Validation epoch plots use fixed decoding thresholds; the heatmap shows the separate post-training validation calibration. Source-level and class-level comparisons use exactly the same frozen test.

In [10]:
def save_figure(figure, filename):
    figure.tight_layout()
    figure.savefig(DIRECTORIES["plots"] / filename, dpi=240, bbox_inches="tight", facecolor="white")
    plt.close(figure)


def grouped_plot(labels, series, title, filename):
    figure, axis = plt.subplots(figsize=(9, 5.5))
    x = np.arange(len(labels))
    width = 0.8 / len(series)
    for index, ((name, values), color) in enumerate(zip(series.items(), ("#6C7A89", "#247BA0", "#E87524"))):
        bars = axis.bar(x + (index - 1) * width, values, width, label=name, color=color)
        axis.bar_label(bars, labels=[f"{v:.3f}" for v in values], padding=3, fontsize=9)
    axis.set_xticks(x, labels)
    axis.set_ylim(0, 1.10)
    axis.set_ylabel("Metric value")
    axis.set_title(title, pad=14)
    axis.grid(axis="y", alpha=0.2)
    axis.set_axisbelow(True)
    axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=3, frameon=False)
    save_figure(figure, filename)


if RUN_MODE == "run":
    plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
    history_frame = pd.DataFrame(history)
    for column, filename, title, ylabel in (
        ("train_loss", "training_loss_v3.png", "V3: weighted ordinal training loss", "Corrective ordinal BCE loss"),
        ("validation_macro_f1", "validation_macro_f1_v3.png", "V3: validation macro-F1 (default thresholds)", "Macro-F1"),
        ("validation_high_recall", "validation_high_recall_v3.png", "V3: validation HIGH recall (default thresholds)", "HIGH recall"),
    ):
        plotted = history_frame.loc[history_frame[column].notna(), ["epoch", column]]
        figure, axis = plt.subplots(figsize=(8, 5))
        axis.plot(plotted.epoch, plotted[column], "o-", color="#247BA0")
        if column != "train_loss":
            chosen = history_frame.loc[history_frame.epoch == best_epoch].iloc[0]
            axis.scatter([best_epoch], [chosen[column]], marker="*", s=180, color="#E87524",
                         zorder=5, label=f"Selected epoch {best_epoch}")
            axis.legend()
        axis.set(xlabel="Epoch (0 = initialized ordinal head)", ylabel=ylabel, title=title)
        axis.set_xticks(history_frame.epoch)
        axis.grid(alpha=0.25)
        save_figure(figure, filename)
        plotted.to_csv(DIRECTORIES["plots"] / filename.replace(".png", ".csv"), index=False)

    heatmap = calibration_grid.pivot(index="t2", columns="t1", values="macro_f1")
    figure, axis = plt.subplots(figsize=(8, 6.5))
    image = axis.imshow(heatmap.to_numpy(), origin="lower", cmap="viridis", aspect="auto")
    ticks = np.arange(0, len(THRESHOLD_GRID), 2)
    axis.set_xticks(ticks, [f"{THRESHOLD_GRID[i]:.2f}" for i in ticks])
    axis.set_yticks(ticks, [f"{THRESHOLD_GRID[i]:.2f}" for i in ticks])
    axis.scatter([list(heatmap.columns).index(calibrated_thresholds[0])],
                 [list(heatmap.index).index(calibrated_thresholds[1])], marker="*", s=180,
                 color="white", edgecolors="black", label="Selected thresholds")
    axis.set(xlabel="t1: score >= 1", ylabel="t2: score >= 2", title="V3: validation-only threshold calibration")
    axis.legend(loc="lower right")
    figure.colorbar(image, ax=axis, label="Validation macro-F1")
    save_figure(figure, "threshold_calibration_heatmap.png")
    calibration_grid.to_csv(DIRECTORIES["plots"] / "threshold_calibration_heatmap.csv", index=False)

    overall_rows = [{"model": name, **{key: values[key] for key in
                    ("accuracy", "macro_f1", "balanced_accuracy", "ordinal_mae")}}
                    for name, values in comparison_metrics.items()]
    pd.DataFrame(overall_rows).to_csv(DIRECTORIES["plots"] / "overall_metrics_comparison_v1_v2_v3.csv", index=False)
    grouped_plot(["Accuracy", "Macro-F1", "Balanced accuracy"],
        {name: [values[k] for k in ("accuracy", "macro_f1", "balanced_accuracy")]
         for name, values in comparison_metrics.items()},
        "Frozen test (140 images): V1 / V2 / V3", "overall_metrics_comparison_v1_v2_v3.png")
    class_rows = [{"model": name, "class": label, **values["per_class"][label]}
                  for name, values in comparison_metrics.items() for label in CLASSES]
    pd.DataFrame(class_rows).to_csv(DIRECTORIES["plots"] / "per_class_metrics_comparison_v1_v2_v3.csv", index=False)
    for metric in ("f1", "recall"):
        grouped_plot([c.upper() for c in CLASSES],
            {name: [values["per_class"][c][metric] for c in CLASSES] for name, values in comparison_metrics.items()},
            f"Frozen test: per-class {metric.upper()}", f"per_class_{metric}_comparison_v1_v2_v3.png")
    sources = {"ComplexDataLab/OpenFake": "OpenFake", "saberzl/SID_Set": "SID-Set", "RRDataset": "RRDataset"}
    source_rows = [{"model": name, "source_dataset": source, **{key: values[key] for key in
                   ("support", "accuracy", "macro_f1", "balanced_accuracy", "ordinal_mae")}}
                   for name, result in comparison_metrics.items() for source, values in result["by_source_dataset"].items()]
    pd.DataFrame(source_rows).to_csv(DIRECTORIES["plots"] / "per_source_metrics_comparison_v1_v2_v3.csv", index=False)
    grouped_plot(list(sources.values()),
        {name: [values["by_source_dataset"][s]["macro_f1"] for s in sources] for name, values in comparison_metrics.items()},
        "Frozen test: macro-F1 by source", "per_source_macro_f1_comparison_v1_v2_v3.png")

    matrix_max = max(np.asarray(v["confusion_matrix"]).max() for v in comparison_metrics.values())
    for name, values in comparison_metrics.items():
        matrix = np.asarray(values["confusion_matrix"])
        figure, axis = plt.subplots(figsize=(6.5, 5.5))
        image = axis.imshow(matrix, cmap="Blues", vmin=0, vmax=matrix_max)
        axis.set_xticks(range(3), [c.upper() for c in CLASSES])
        axis.set_yticks(range(3), [c.upper() for c in CLASSES])
        axis.set(xlabel="Predicted sensitivity", ylabel="Human sensitivity target",
                 title=f"{name}: frozen-test confusion matrix (n=140)")
        for i in range(3):
            for j in range(3):
                axis.text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=14,
                          color="white" if matrix[i, j] > matrix_max / 2 else "black")
        figure.colorbar(image, ax=axis, label="Images")
        filename = f"confusion_matrix_{name.lower()}.png"
        save_figure(figure, filename)
        pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(DIRECTORIES["plots"] / filename.replace(".png", ".csv"), index_label="true_label")
    print("Saved all 11 report plots and their numerical tables.", flush=True)

Saved all 11 report plots and their numerical tables.


## Optional final corpus export only if V3 is promoted

Read the canonical 4,499-image corpus, preserving the known within-RRDataset duplicate's two source paths. Authenticity comes solely from verified source metadata. The selected calibrated decoder supplies sensitivity. The 400 final human reviews and the 701 original human annotations are preserved separately; no existing autoannotation file is modified.

In [11]:
final_dataset_paths = []
if RUN_MODE == "run" and final_model == "V3" and EXPORT_IF_V3_WINS:
    corpus = read_csv(DATA_ROOT / "corpus_manifest.csv")
    require(len(corpus) == len({r["content_hash"] for r in corpus}) == 4499, "Corpus count/identity mismatch")
    corpus_hashes = {r["content_hash"] for r in corpus}
    require(len(reviews_by_hash) == 400 and set(reviews_by_hash) <= corpus_hashes, "Human reviews missing from corpus")
    require(len(master_by_hash) == 701 and set(master_by_hash) <= corpus_hashes, "Original human data missing from corpus")
    for row in corpus:
        path = DATA_ROOT / row["image_path"]
        require(path.is_file() and sha256(path) == row["content_hash"], f"Corpus image integrity failure: {path}")
        require(row["source_label"] in {"real", "fake"} and "verified" in row["source_label_provenance"],
                f"Unverified corpus authenticity: {row['content_hash']}")
        require(row["source_dataset"] in sources, "Unknown corpus source")
    class CorpusImages(Dataset):
        def __len__(self):
            return len(corpus)
        def __getitem__(self, index):
            with Image.open(DATA_ROOT / corpus[index]["image_path"]) as image:
                rgb = ImageOps.exif_transpose(image).convert("RGB")
            return eval_transform(rgb), index
    corpus_loader = DataLoader(CorpusImages(), batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    exported = []
    model.eval()
    with torch.inference_mode():
        for batch, (images, indices) in enumerate(corpus_loader, 1):
            raw = torch.sigmoid(model(images.to(DEVICE, non_blocking=True))).cpu().numpy()
            corrected, violations = monotonic_probabilities(raw)
            predictions = decode(corrected, *calibrated_thresholds)
            for j, index in enumerate(indices.tolist()):
                source = corpus[index]
                digest = source["content_hash"]
                review = reviews_by_hash.get(digest)
                original = master_by_hash.get(digest)
                if review:
                    require(review["source_label"] == source["source_label"]
                            and review["source_dataset"] == source["source_dataset"], "Reviewed source mismatch")
                label = review["human_final_label"] if review else None
                original_label = original["final_sensitivity_level"] if original else None
                exported.append({"image_id": digest, "content_hash": digest,
                    "image_path": (PACKAGE / source["image_path"]).as_posix(),
                    "source_dataset": source["source_dataset"], "source_label": source["source_label"],
                    "real_fake_label": 0 if source["source_label"] == "real" else 1,
                    "predicted_sensitivity_score": int(predictions[j]),
                    "predicted_sensitivity_label": CLASSES[int(predictions[j])],
                    "p_ge_1_raw": float(raw[j, 0]), "p_ge_2_raw": float(raw[j, 1]),
                    "p_ge_1": float(corrected[j, 0]), "p_ge_2": float(corrected[j, 1]),
                    "monotonicity_corrected": bool(violations[j]),
                    "t1": calibrated_thresholds[0], "t2": calibrated_thresholds[1],
                    "model_version": MODEL_VERSION, "checkpoint_sha256": checkpoint_hash,
                    "human_review_available": bool(review),
                    "human_sensitivity_score": CLASS_TO_INDEX[label] if label else None,
                    "human_sensitivity_label": label, "human_final_label": label,
                    "original_human_annotation_available": bool(original),
                    "original_human_sensitivity_score": CLASS_TO_INDEX[original_label] if original_label else None,
                    "original_human_sensitivity_label": original_label,
                    "evaluation_role": "frozen_test" if digest in set(test_frame.content_hash) else "other",
                    "source_label_provenance": source["source_label_provenance"],
                    "source_paths_json": source["source_paths_json"]})
            if batch % 25 == 0:
                print(f"V3 annotated {len(exported)}/4499 corpus images", flush=True)
    require(len(exported) == len({r["image_id"] for r in exported}) == 4499, "Export duplicates or incomplete rows")
    require(len({r["image_path"] for r in exported}) == 4499, "Duplicate physical export paths")
    require(sum(r["human_review_available"] for r in exported) == 400, "Human-review annotation loss")
    require(all(r["real_fake_label"] in {0, 1} and r["predicted_sensitivity_score"] in {0, 1, 2}
                and r["predicted_sensitivity_label"] == CLASSES[r["predicted_sensitivity_score"]]
                and 0 <= r["p_ge_2"] <= r["p_ge_1"] <= 1 for r in exported), "Invalid export annotations")
    require(all(r["human_sensitivity_label"] == reviews_by_hash[r["image_id"]]["human_final_label"]
                for r in exported if r["human_review_available"]), "Human labels changed in export")
    csv_path = DIRECTORIES["final_dataset"] / "sensifake_autoannotated_v3.csv"
    jsonl_path = DIRECTORIES["final_dataset"] / "sensifake_autoannotated_v3.jsonl"
    with csv_path.open("x", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(exported[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows({k: str(v).lower() if isinstance(v, bool) else v for k, v in row.items()} for row in exported)
    with jsonl_path.open("x", encoding="utf-8") as stream:
        for row in exported:
            stream.write(json.dumps(row, allow_nan=False, ensure_ascii=False) + "\n")
    audit = {"selected_model": "V3", "model_version": MODEL_VERSION,
        "checkpoint": BEST_CHECKPOINT.relative_to(REPO_ROOT).as_posix(), "checkpoint_sha256": checkpoint_hash,
        "total_images": len(exported), "unique_image_ids": 4499,
        "thresholds": {"t1": calibrated_thresholds[0], "t2": calibrated_thresholds[1]},
        "source_dataset_counts": dict(Counter(r["source_dataset"] for r in exported)),
        "real_fake_counts": {str(a): sum(r["real_fake_label"] == a for r in exported) for a in (0, 1)},
        "sensitivity_score_counts": {str(s): sum(r["predicted_sensitivity_score"] == s for r in exported) for s in (0, 1, 2)},
        "real_fake_by_sensitivity": {str(a): {str(s): sum(r["real_fake_label"] == a and r["predicted_sensitivity_score"] == s
                                                       for r in exported) for s in (0, 1, 2)} for a in (0, 1)},
        "human_reviewed_images": 400, "unreviewed_images": 4099,
        "original_human_annotation_images": sum(r["original_human_annotation_available"] for r in exported),
        "monotonicity_corrections": sum(r["monotonicity_corrected"] for r in exported),
        "deduplication_reason": "4500 selected source files yield 4499 SHA-256 identities; RRDataset real_008773.jpg and real_009566.jpg are byte-identical",
        "csv_sha256": sha256(csv_path), "jsonl_sha256": sha256(jsonl_path)}
    audit_path = DIRECTORIES["final_dataset"] / "sensifake_autoannotated_v3_audit.json"
    save_json(audit_path, audit)
    final_dataset_paths = [p.relative_to(REPO_ROOT).as_posix() for p in (csv_path, jsonl_path, audit_path)]
    print("V3 corpus export completed:", json.dumps(audit, indent=2), flush=True)
elif RUN_MODE == "run":
    print("Full corpus inference skipped:", "V2 remains final" if final_model == "V2" else "optional V3 export disabled", flush=True)

V3 annotated 800/4499 corpus images


V3 annotated 1600/4499 corpus images


V3 annotated 2400/4499 corpus images


V3 annotated 3200/4499 corpus images


V3 annotated 4000/4499 corpus images


V3 corpus export completed: {
  "selected_model": "V3",
  "model_version": "resnet50_layer4_fc_weighted_ordinal_v3",
  "checkpoint": "autoannotator-final version/output/checkpoints/best_model_ordinal_v3.pt",
  "checkpoint_sha256": "48f6495d4a62ded10d98c3835ffcd33c8f5c653e5df19cb03aef62582060e9b4",
  "total_images": 4499,
  "unique_image_ids": 4499,
  "thresholds": {
    "t1": 0.5,
    "t2": 0.44
  },
  "source_dataset_counts": {
    "saberzl/SID_Set": 1500,
    "RRDataset": 1499,
    "ComplexDataLab/OpenFake": 1500
  },
  "real_fake_counts": {
    "0": 2249,
    "1": 2250
  },
  "sensitivity_score_counts": {
    "0": 1177,
    "1": 3112,
    "2": 210
  },
  "real_fake_by_sensitivity": {
    "0": {
      "0": 602,
      "1": 1607,
      "2": 40
    },
    "1": {
      "0": 575,
      "1": 1505,
      "2": 170
    }
  },
  "human_reviewed_images": 400,
  "unreviewed_images": 4099,
  "original_human_annotation_images": 701,
  "monotonicity_corrections": 5,
  "deduplication_reason": "4500 

## Final saved summary

The summary reports actual execution results and paths. No reported V3 metrics exist until this notebook has been run; if V2 stays best, the V3 final_dataset directory remains empty.

In [12]:
if RUN_MODE == "run":
    check_inputs_unchanged()
    summary = {"gpu_used": DEVICE.type == "cuda", "device": str(DEVICE), "best_epoch": best_epoch,
        "t1": calibrated_thresholds[0], "t2": calibrated_thresholds[1],
        "validation_macro_f1_default_thresholds": best_score,
        "validation_macro_f1_calibrated": calibrated_validation_metrics["macro_f1"],
        "v3_test": test_metrics, "comparison": comparison_metrics,
        "source_level_best_macro_f1": {s: max(comparison_metrics, key=lambda name: comparison_metrics[name]["by_source_dataset"][s]["macro_f1"])
                                        for s in test_metrics["by_source_dataset"]},
        "v3_beats_v2_macro_f1": macro_gain > 1e-12, "final_selection": selection,
        "output_directory": OUTPUT.relative_to(REPO_ROOT).as_posix(),
        "plots": [p.relative_to(REPO_ROOT).as_posix() for p in sorted(DIRECTORIES["plots"].glob("*.png"))],
        "predictions": [p.relative_to(REPO_ROOT).as_posix() for p in sorted(DIRECTORIES["predictions"].glob("*.csv"))],
        "final_dataset": final_dataset_paths,
        "protected_inputs_unchanged": True}
    save_json(DIRECTORIES["metrics"] / "final_summary.json", summary)
    print(json.dumps(summary, indent=2), flush=True)

{
  "gpu_used": true,
  "device": "cuda",
  "best_epoch": 12,
  "t1": 0.5,
  "t2": 0.44,
  "validation_macro_f1_default_thresholds": 0.6626861767388457,
  "validation_macro_f1_calibrated": 0.6854613668257453,
  "v3_test": {
    "support": 140,
    "class_order": [
      "low",
      "medium",
      "high"
    ],
    "accuracy": 0.7571428571428571,
    "macro_f1": 0.7086106069772132,
    "balanced_accuracy": 0.709888564318944,
    "ordinal_mae": 0.24285714285714285,
    "high_recall": 0.6666666666666666,
    "high_f1": 0.631578947368421,
    "per_class": {
      "low": {
        "precision": 0.8571428571428571,
        "recall": 0.5769230769230769,
        "f1": 0.6896551724137931,
        "support": 52
      },
      "medium": {
        "precision": 0.7368421052631579,
        "recall": 0.8860759493670886,
        "f1": 0.8045977011494253,
        "support": 79
      },
      "high": {
        "precision": 0.6,
        "recall": 0.6666666666666666,
        "f1": 0.631578947368421,
    